# Clinical Trial Firms with AI Patents Analysis

**Objective:** Compute the percentage of companies involved in clinical trials that hold at least one AI-related patent.

**AI Patent Definition:** Patents with CPC code starting with "G06N"

## Data Pipeline

1. **Clinical Trials** → obtain GVKEY for firms involved in clinical trials
2. **DISCERN** → use GVKEY to obtain:
   - `permno_adj` from `discern_firm_panel_1980_2021`
   - `patent_id` from `discern_pat_grant_1980_2021` via `permno_adj`
3. **PatentsView** → use `patent_id` to obtain CPC classifications
4. **Filter** patents where CPC starts with "G06N"

## Time Window

- Use firm fiscal year (`fyear`) from `discern_firm_panel_1980_2021`
- Restrict analysis to firms with `fyear` between 2000 and 2021
- Note: DISCERN data ends in 2021, so years 2022–2023 are excluded

---

## Setup and Imports

In [99]:
import pandas as pd
import numpy as np
import requests
import time
from pathlib import Path
import warnings
from tqdm.auto import tqdm
import json

warnings.filterwarnings('ignore')

# Enable progress bar for pandas operations
tqdm.pandas()

print("Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries imported successfully
Pandas version: 2.3.3
NumPy version: 2.3.5


## Step 1: Load Clinical Trial Data

Extract unique firms (GVKEYs) associated with clinical trials.

In [100]:
# Define paths
BASE_DIR = Path('..')
CLINICAL_TRIALS_PATH = BASE_DIR / 'Task2 Outdated' / 'clinical_trial_sample (1).csv'
DISCERN_DIR = BASE_DIR / 'Task2 Outdated' / 'Task2 Outdated V2' / 'DISCERN 2_0_1' / 'output_files' / 'stata_files'

print(f"Clinical trials path: {CLINICAL_TRIALS_PATH}")
print(f"DISCERN directory: {DISCERN_DIR}")
print(f"Clinical trials file exists: {CLINICAL_TRIALS_PATH.exists()}")
print(f"DISCERN directory exists: {DISCERN_DIR.exists()}")

Clinical trials path: ../Task2 Outdated/clinical_trial_sample (1).csv
DISCERN directory: ../Task2 Outdated/Task2 Outdated V2/DISCERN 2_0_1/output_files/stata_files
Clinical trials file exists: True
DISCERN directory exists: True


In [101]:
# Load clinical trials data
print("Loading clinical trials data...")
clinical_trials = pd.read_csv(CLINICAL_TRIALS_PATH, encoding='utf-8-sig')

print(f"\nClinical trials dataset shape: {clinical_trials.shape}")
print(f"\nColumns: {clinical_trials.columns.tolist()}")
print(f"\nFirst few rows:")
clinical_trials.head()

Loading clinical trials data...

Clinical trials dataset shape: (9428, 8)

Columns: ['nct_id', 'brief_title', 'overall_status', 'sponsor_name', 'gvkey_sponsor', 'phase_number', 'start_date', 'start_year']

First few rows:


,nct_id,brief_title,overall_status,sponsor_name,gvkey_sponsor,phase_number,start_date,start_year
0,NCT00175851,Open Label Trial to Study the Long-term Safety...,Withdrawn,UCB Pharma,24454,3,2008-05-01,2008
1,NCT00359632,Study to Evaluate Eye Function in Patients Tak...,Terminated,Pfizer,8530,3,2008-11-01,2008
2,NCT00415155,A Study of LY2181308 in Patients With Advanced...,Withdrawn,Eli Lilly and Company,6730,2,2008-08-01,2008
3,NCT00422110,A Study to Evaluate the Efficacy and Safety of...,Withdrawn,UCB Pharma,24454,3,2008-05-01,2008
4,NCT00422422,"Open-label, Pharmacokinetic, Safety and Effica...",Completed,UCB Pharma,24454,2,2011-07-01,2011


In [102]:
# Extract unique GVKEYs from clinical trials
# Remove any missing GVKEYs

# CRITICAL FIX: Convert to 6-digit zero-padded strings to match DISCERN format
# DISCERN stores GVKEYs as '001010', '008530', etc. (6-digit strings)
# Clinical trials stores them as integers: 1010, 8530, etc.

print("Converting clinical trial GVKEYs to match DISCERN format...")
print(f"Before: Sample GVKEYs = {clinical_trials['gvkey_sponsor'].dropna().head().tolist()}")

# Convert to 6-digit zero-padded strings
clinical_trials['gvkey_sponsor'] = clinical_trials['gvkey_sponsor'].apply(
    lambda x: str(int(x)).zfill(6) if pd.notna(x) else x
)

ct_gvkeys = clinical_trials['gvkey_sponsor'].dropna().unique()

print(f"After:  Sample GVKEYs = {list(ct_gvkeys[:5])}")
print(f"✓ GVKEYs formatted as 6-digit strings (e.g., 8530 → '008530')")

print(f"\nTotal unique GVKEYs in clinical trials: {len(ct_gvkeys)}")
print(f"Sample GVKEYs: {list(ct_gvkeys[:10])}")

Converting clinical trial GVKEYs to match DISCERN format...
Before: Sample GVKEYs = [24454, 8530, 6730, 24454, 24454]
After:  Sample GVKEYs = ['024454', '008530', '006730', '001602', '031628']
✓ GVKEYs formatted as 6-digit strings (e.g., 8530 → '008530')

Total unique GVKEYs in clinical trials: 673
Sample GVKEYs: ['024454', '008530', '006730', '001602', '031628', '038821', '009775', '007392', '027845', '245207']


## Step 2: Load DISCERN Firm Panel Data

Map GVKEY → permno_adj using `discern_firm_panel_1980_2021` with fiscal year filtering (2000-2021).

In [103]:
# Load DISCERN firm panel data
print("Loading DISCERN firm panel data...")
firm_panel_path = DISCERN_DIR / 'discern_firm_panel_1980_2021.dta'

# Load Stata file
firm_panel = pd.read_stata(firm_panel_path)

print(f"Firm panel shape: {firm_panel.shape}")
print(f"\nColumns: {firm_panel.columns.tolist()}")
print(f"\nFirst few rows:")
firm_panel.head()

Loading DISCERN firm panel data...
Firm panel shape: (111082, 7)

Columns: ['permno_adj', 'gvkey', 'fyear', 'pat_grant_fyear', 'pat_app_fyear', 'pub_fyear', 'name_permno_adj']

First few rows:


,permno_adj,gvkey,fyear,pat_grant_fyear,pat_app_fyear,pub_fyear,name_permno_adj
0,10006,001010,1980,59.0,0.0,1.0,ACF IND INC
1,10006,001010,1981,47.0,0.0,0.0,ACF IND INC
2,10006,001010,1982,24.0,0.0,0.0,ACF IND INC
3,10006,001010,1983,28.0,0.0,0.0,ACF IND INC
4,10006,001010,1984,20.0,0.0,0.0,ACF IND INC


In [104]:
# Check data types and ranges
print("Data summary:")
print(f"Fiscal year range: {firm_panel['fyear'].min()} to {firm_panel['fyear'].max()}")
print(f"Unique GVKEYs: {firm_panel['gvkey'].nunique()}")
print(f"Unique permno_adj: {firm_panel['permno_adj'].nunique()}")
print(f"\nData types:\n{firm_panel.dtypes}")

# NOTE: DISCERN GVKEYs are already in string format (e.g., '001010', '008530')
# We will match against clinical trial GVKEYs that we converted to the same format
print(f"\nGVKEY format in DISCERN: {firm_panel['gvkey'].dtype}")
print(f"Sample GVKEYs from firm panel: {firm_panel['gvkey'].unique()[:10].tolist()}")

Data summary:
Fiscal year range: 1980 to 2021
Unique GVKEYs: 8198
Unique permno_adj: 8036

Data types:
permno_adj           int32
gvkey               object
fyear                int16
pat_grant_fyear    float64
pat_app_fyear      float64
pub_fyear          float64
name_permno_adj     object
dtype: object

GVKEY format in DISCERN: object
Sample GVKEYs from firm panel: ['001010', '009636', '012622', '011907', '011908', '004008', '011903', '012825', '012595', '011948']


In [105]:
# Filter for:
# 1. Fiscal year between 2000 and 2021
# 2. GVKEYs that are in our clinical trials dataset
# 3. Non-missing permno_adj

print("Filtering firm panel data...")
print(f"\nDiagnostic info BEFORE filtering:")
print(f"- ct_gvkeys dtype: {type(ct_gvkeys[0]) if len(ct_gvkeys) > 0 else 'N/A'}")
print(f"- firm_panel['gvkey'] dtype: {firm_panel['gvkey'].dtype}")
print(f"- Sample ct_gvkeys: {list(ct_gvkeys[:5])}")
print(f"- Sample firm_panel gvkeys: {firm_panel['gvkey'].unique()[:5].tolist()}")
print(f"- Any overlap? {len(set(ct_gvkeys) & set(firm_panel['gvkey'].unique()))} GVKEYs")

firm_panel_filtered = firm_panel[
    (firm_panel['fyear'] >= 2000) & 
    (firm_panel['fyear'] <= 2021) &
    (firm_panel['gvkey'].isin(ct_gvkeys)) &
    (firm_panel['permno_adj'].notna())
].copy()

print(f"\nFiltered firm panel shape: {firm_panel_filtered.shape}")
print(f"Unique GVKEYs matched: {firm_panel_filtered['gvkey'].nunique()}")
print(f"Unique permno_adj matched: {firm_panel_filtered['permno_adj'].nunique()}")
print(f"\nMatch rate: {firm_panel_filtered['gvkey'].nunique() / len(ct_gvkeys) * 100:.2f}%")

Filtering firm panel data...

Diagnostic info BEFORE filtering:
- ct_gvkeys dtype: <class 'str'>
- firm_panel['gvkey'] dtype: object
- Sample ct_gvkeys: ['024454', '008530', '006730', '001602', '031628']
- Sample firm_panel gvkeys: ['001010', '009636', '012622', '011907', '011908']
- Any overlap? 344 GVKEYs

Filtered firm panel shape: (3534, 7)
Unique GVKEYs matched: 341
Unique permno_adj matched: 335

Match rate: 50.67%


In [106]:
# Get unique GVKEY-permno_adj mappings
# A GVKEY can map to multiple permno_adj over time
gvkey_permno_mapping = firm_panel_filtered[['gvkey', 'permno_adj']].drop_duplicates()

print(f"Unique GVKEY-permno_adj pairs: {len(gvkey_permno_mapping)}")
print(f"\nSample mappings:")
gvkey_permno_mapping.head(10)

Unique GVKEY-permno_adj pairs: 342

Sample mappings:


,gvkey,permno_adj
580,012180,10143
954,012181,10200
1248,179598,10258
1721,012250,10363
6207,013365,11415
6792,013599,11552
7193,013786,11636
7648,001259,11739
9163,184884,12085
9185,184114,12090


## Step 3: Load DISCERN Patent Grant Data

Map permno_adj → patent_id using `discern_pat_grant_1980_2021`.

In [107]:
# Load DISCERN patent grant data
print("Loading DISCERN patent grant data...")
patent_grant_path = DISCERN_DIR / 'discern_pat_grant_1980_2021.dta'

# This file can be large, so we'll load it in chunks if needed
print("Note: This may take a few minutes for large files...")
patent_grants = pd.read_stata(patent_grant_path)

print(f"\nPatent grants shape: {patent_grants.shape}")
print(f"\nColumns: {patent_grants.columns.tolist()}")
print(f"\nFirst few rows:")
patent_grants.head()

Loading DISCERN patent grant data...
Note: This may take a few minutes for large files...

Patent grants shape: (1874327, 9)

Columns: ['patent_id', 'patent_date', 'assignee_name', 'fyear', 'name_std', 'id_name', 'sample', 'permno_adj', 'name_permno_adj']

First few rows:


,patent_id,patent_date,assignee_name,fyear,name_std,id_name,sample,permno_adj,name_permno_adj
0,4206709,1980-06-10,"ACF Industries, Incorporated",1980,ACF IND INC,36,U,10006,ACF IND INC
1,4207822,1980-06-17,"ACF Industries, Incorporated",1980,ACF IND INC,36,U,10006,ACF IND INC
2,4208035,1980-06-17,"ACF Industries, Incorporated",1980,ACF IND INC,36,U,10006,ACF IND INC
3,4208544,1980-06-17,"ACF Industries, Incorporated",1980,ACF IND INC,36,U,10006,ACF IND INC
4,4209036,1980-06-24,"ACF Industries, Incorporated",1980,ACF IND INC,36,U,10006,ACF IND INC


In [108]:
# Check data summary
print("Patent grants data summary:")
print(f"Total patents: {len(patent_grants)}")
print(f"Unique permno_adj: {patent_grants['permno_adj'].nunique()}")
print(f"Unique patent_id: {patent_grants['patent_id'].nunique()}")

if 'appyear' in patent_grants.columns:
    print(f"Application year range: {patent_grants['appyear'].min()} to {patent_grants['appyear'].max()}")
if 'gyear' in patent_grants.columns:
    print(f"Grant year range: {patent_grants['gyear'].min()} to {patent_grants['gyear'].max()}")

Patent grants data summary:
Total patents: 1874327
Unique permno_adj: 5681
Unique patent_id: 1865424


In [109]:
# Filter for permno_adj that are in our clinical trial firms
ct_permno = gvkey_permno_mapping['permno_adj'].unique()

print(f"\nFiltering patents for {len(ct_permno)} clinical trial permno_adj values...")
ct_patents = patent_grants[
    patent_grants['permno_adj'].isin(ct_permno)
].copy()

print(f"Patents from clinical trial firms: {len(ct_patents)}")
print(f"Unique patent_id: {ct_patents['patent_id'].nunique()}")
print(f"Unique permno_adj: {ct_patents['permno_adj'].nunique()}")


Filtering patents for 335 clinical trial permno_adj values...
Patents from clinical trial firms: 76825
Unique patent_id: 76610
Unique permno_adj: 304


In [110]:
# Merge patent data with GVKEY mapping
ct_patents_with_gvkey = ct_patents.merge(
    gvkey_permno_mapping,
    on='permno_adj',
    how='left'
)

print(f"Patents with GVKEY mapping: {len(ct_patents_with_gvkey)}")
print(f"Unique GVKEYs with patents: {ct_patents_with_gvkey['gvkey'].nunique()}")
print(f"\nSample data:")
ct_patents_with_gvkey[['patent_id', 'permno_adj', 'gvkey']].head(10)

Patents with GVKEY mapping: 77111
Unique GVKEYs with patents: 311

Sample data:


,patent_id,permno_adj,gvkey
0,5190858,10143,012180
1,5217896,10143,012180
2,5262319,10143,012180
3,5262523,10143,012180
4,5401638,10143,012180
5,5443956,10143,012180
6,5580722,10143,012180
7,5604107,10143,012180
8,5635489,10143,012180
9,5665543,10143,012180


## Step 4: Retrieve CPC Codes from PatentsView Bulk Data

**Note:** The PatentsView API was deprecated on May 1, 2025. We now use bulk data downloads instead.

We'll download the CPC classification data and merge it with our patent IDs.

In [111]:
# Get unique patent IDs
unique_patent_ids = ct_patents_with_gvkey['patent_id'].unique()
print(f"Total unique patents to query: {len(unique_patent_ids)}")

# Convert patent IDs to strings for matching
unique_patent_ids_str = set(str(pid) for pid in unique_patent_ids)
print(f"Sample patent IDs: {list(unique_patent_ids[:10])}")

Total unique patents to query: 76610
Sample patent IDs: ['5190858', '5217896', '5262319', '5262523', '5401638', '5443956', '5580722', '5604107', '5635489', '5665543']


In [112]:
def download_and_load_cpc_data(patent_ids_set, cache_file='cpc_codes_checkpoint.csv'):
    """
    Download and load CPC data from PatentsView bulk downloads.
    
    The PatentsView API was deprecated on May 1, 2025. We now use bulk data files.
    
    Parameters:
    - patent_ids_set: set of patent IDs (strings) to filter for
    - cache_file: path to cache file to avoid re-downloading
    
    Returns:
    - DataFrame with columns: patent_id, cpc_group
    """
    
    # Check if cached file exists
    cache_path = Path(cache_file)
    if cache_path.exists():
        print(f"Loading CPC data from cache: {cache_file}")
        cpc_df = pd.read_csv(cache_path)
        print(f"Loaded {len(cpc_df)} CPC records from cache")
        return cpc_df
    
    print("Downloading CPC data from PatentsView bulk downloads...")
    print("Note: This file is large (~4GB compressed, ~8GB uncompressed)")
    print("This may take 10-30 minutes to download and process.")
    
    # PatentsView bulk download URL for granted patents CPC data
    cpc_url = "https://s3.amazonaws.com/data.patentsview.org/download/g_cpc_current.tsv.zip"
    
    try:
        # Download with progress
        print(f"\nDownloading from: {cpc_url}")
        print("Please wait, this is a large file...")
        response = requests.get(cpc_url, stream=True, timeout=600)
        response.raise_for_status()
        
        # Save to temp file
        import zipfile
        import io
        
        print("Download complete. Extracting...")
        zip_file = zipfile.ZipFile(io.BytesIO(response.content))
        
        # Get the TSV filename
        tsv_filename = [f for f in zip_file.namelist() if f.endswith('.tsv')][0]
        print(f"Extracting {tsv_filename}...")
        
        # Read TSV in chunks and filter for our patent IDs
        print(f"\nLoading and filtering CPC data for {len(patent_ids_set)} patents...")
        
        filtered_chunks = []
        with zip_file.open(tsv_filename) as f:
            # Read in chunks to manage memory
            # Key columns: patent_id, cpc_group
            chunk_iter = pd.read_csv(
                f, 
                sep='\t', 
                chunksize=100000, 
                dtype={'patent_id': str},
                usecols=['patent_id', 'cpc_group']  # Only load columns we need
            )
            
            for i, chunk in enumerate(chunk_iter):
                # Filter for our patents
                filtered = chunk[chunk['patent_id'].isin(patent_ids_set)]
                if len(filtered) > 0:
                    filtered_chunks.append(filtered)
                
                if (i + 1) % 10 == 0:
                    total_found = sum(len(c) for c in filtered_chunks)
                    print(f"  Processed {(i+1)*100000:,} rows, found {total_found:,} matches...")
        
        if filtered_chunks:
            cpc_df = pd.concat(filtered_chunks, ignore_index=True)
        else:
            print("Warning: No CPC codes found for the provided patent IDs")
            return pd.DataFrame(columns=['patent_id', 'cpc_group'])
        
        print(f"\nFiltered to {len(cpc_df)} CPC records for {cpc_df['patent_id'].nunique()} patents")
        
        # Cache the results
        cpc_df.to_csv(cache_path, index=False)
        print(f"Cached CPC data to {cache_file}")
        
        return cpc_df
        
    except requests.exceptions.RequestException as e:
        print(f"\nError downloading CPC data: {str(e)}")
        print("\nALTERNATIVE: You can download manually from:")
        print("https://patentsview.org/download/data-download-tables")
        print("Look for 'g_cpc_current.tsv.zip' under Granted Patents")
        return pd.DataFrame(columns=['patent_id', 'cpc_group'])
    
    except Exception as e:
        print(f"\nError processing CPC data: {str(e)}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame(columns=['patent_id', 'cpc_group'])

print("Function defined. Ready to download and process CPC codes.")

Function defined. Ready to download and process CPC codes.


In [113]:
# Download and load CPC codes for all patents
# Note: This will download ~4GB of data on first run, then cache it locally
# Subsequent runs will use the cached data

if len(unique_patent_ids_str) > 0:
    print(f"Starting CPC code retrieval for {len(unique_patent_ids_str)} patents...")
    print("This may take 10-30 minutes on first run (downloading bulk data).")
    print("Subsequent runs will be much faster (using cached data).\n")

    cpc_data = download_and_load_cpc_data(
        unique_patent_ids_str,
        cache_file='cpc_codes_checkpoint.csv'
    )

    print(f"\nCPC data summary:")
    print(f"- Total CPC records: {len(cpc_data)}")
    print(f"- Unique patents with CPC data: {cpc_data['patent_id'].nunique()}")
    print(f"- Coverage rate: {cpc_data['patent_id'].nunique() / len(unique_patent_ids_str) * 100:.2f}%")
else:
    print("No patents to query. Skipping CPC download.")
    cpc_data = pd.DataFrame(columns=['patent_id', 'cpc_group'])

Starting CPC code retrieval for 76610 patents...
This may take 10-30 minutes on first run (downloading bulk data).
Subsequent runs will be much faster (using cached data).

Note: This file is large (~4GB compressed, ~8GB uncompressed)
This may take 10-30 minutes to download and process.

Please wait, this is a large file...
Download complete. Extracting...
Extracting g_cpc_current.tsv...

Loading and filtering CPC data for 76610 patents...
  Processed 1,000,000 rows, found 1,841 matches...
  Processed 2,000,000 rows, found 8,426 matches...
  Processed 3,000,000 rows, found 16,098 matches...
  Processed 4,000,000 rows, found 24,702 matches...
  Processed 5,000,000 rows, found 34,821 matches...
  Processed 6,000,000 rows, found 50,832 matches...
  Processed 7,000,000 rows, found 69,894 matches...
  Processed 8,000,000 rows, found 92,531 matches...
  Processed 9,000,000 rows, found 112,838 matches...
  Processed 10,000,000 rows, found 135,705 matches...
  Processed 11,000,000 rows, found 

In [114]:
# Display sample CPC data
print("Sample CPC data:")
if len(cpc_data) > 0:
    print(cpc_data.head(20))
    
    print(f"\nCPC code coverage:")
    print(f"- Patents with CPC codes: {cpc_data['patent_id'].nunique()}")
    print(f"- Total CPC classifications: {len(cpc_data)}")
    print(f"- Average CPC codes per patent: {len(cpc_data) / cpc_data['patent_id'].nunique():.2f}")
else:
    print("No CPC data available")

Sample CPC data:
   patent_id    cpc_group
0    4205400    A61F2/389
1    4206069     C11D3/43
2    4206069    C11D17/00
3    4206069    C11D10/04
4    4206069   C11D10/042
5    4206069   C11D10/045
6    4206069  C11D17/0095
7    4206069    C11D17/06
8    4206069     C11D1/10
9    4206069    C11D1/126
10   4206069     C11D1/14
11   4206069    C11D1/146
12   4206069     C11D1/16
13   4206069     C11D1/22
14   4206069     C11D1/29
15   4206069    C11D1/345
16   4206069     C11D1/72
17   4206070    C11D1/825
18   4206070     C11D1/72
19   4206070    C11D1/722

CPC code coverage:
- Patents with CPC codes: 76494
- Total CPC classifications: 806369
- Average CPC codes per patent: 10.54


In [115]:
# IMPORTANT: Delete old cache if it exists from previous runs with wrong format
cache_file_path = Path('cpc_codes_checkpoint.csv')
if cache_file_path.exists():
    # Check if cache is valid by looking at file size
    file_size = cache_file_path.stat().st_size
    if file_size < 1000:  # Less than 1KB means it's likely empty/corrupted
        print(f"Detected invalid cache file (only {file_size} bytes). Deleting...")
        cache_file_path.unlink()
        print("Cache deleted. Will download fresh data.")

print(f"Cache file location: {cache_file_path.absolute()}")
print("Delete this file to force a fresh download on the next run.")

Cache file location: /Users/eddiejung/Desktop/Research /Deliverables/Task2/Task2 Final Deliverable/cpc_codes_checkpoint.csv
Delete this file to force a fresh download on the next run.


### Optional: Manual Download and Load

If automatic download fails, you can manually download the CPC data:

1. Visit https://patentsview.org/download/data-download-tables
2. Download `g_cpc_current.tsv.zip` (granted patents CPC data)
3. Place it in the notebook directory
4. Update the code above to point to your local file

In [116]:
# If you have a manually downloaded CPC file, you can load it here:
# Uncomment and adjust the path as needed

# import zipfile
# 
# cpc_file_path = 'path/to/g_cpc_current.tsv.zip'
# 
# with zipfile.ZipFile(cpc_file_path) as z:
#     with z.open('g_cpc_current.tsv') as f:
#         # Load and filter
#         filtered_chunks = []
#         chunk_iter = pd.read_csv(f, sep='\t', chunksize=100000, dtype={'patent_id': str})
#         
#         for chunk in tqdm(chunk_iter, desc="Processing CPC chunks"):
#             filtered = chunk[chunk['patent_id'].isin(unique_patent_ids_str)]
#             if len(filtered) > 0:
#                 filtered_chunks.append(filtered)
#         
#         if filtered_chunks:
#             cpc_data = pd.concat(filtered_chunks, ignore_index=True)
#             # Process columns as needed
#             print(f"Loaded {len(cpc_data)} CPC records")

## Step 5: Filter Patents with G06N CPC Prefix

Identify patents where any CPC code starts with "G06N".

In [117]:
# Filter for G06N (AI-related) CPC codes
# CPC G06N = "Computing arrangements based on specific computational models"
# Includes: neural networks, machine learning, knowledge-based systems, etc.

print("Filtering for G06N (AI-related) CPC codes...")

# Mark rows where CPC group starts with G06N
cpc_data['is_ai_cpc'] = cpc_data['cpc_group'].str.startswith('G06N', na=False)

print(f"\nTotal CPC records: {len(cpc_data)}")
print(f"AI-related CPC records (G06N): {cpc_data['is_ai_cpc'].sum()}")

if len(cpc_data) > 0:
    print(f"Percentage: {cpc_data['is_ai_cpc'].sum() / len(cpc_data) * 100:.2f}%")
    
    # Show top G06N codes
    if cpc_data['is_ai_cpc'].sum() > 0:
        print("\nTop G06N CPC codes found:")
        print(cpc_data[cpc_data['is_ai_cpc']]['cpc_group'].value_counts().head(10))
else:
    print("Percentage: N/A (no CPC data to analyze)")

Filtering for G06N (AI-related) CPC codes...

Total CPC records: 806369
AI-related CPC records (G06N): 122
Percentage: 0.02%

Top G06N CPC codes found:
cpc_group
G06N20/00     21
G06N3/09      13
G06N3/0464    13
G06N3/08      13
G06N5/04       9
G06N3/045      9
G06N7/01       6
G06N5/046      4
G06N5/01       3
G06N5/025      3
Name: count, dtype: int64


In [118]:
# Get unique patents that have at least one G06N CPC code
ai_patent_ids = cpc_data[cpc_data['is_ai_cpc']]['patent_id'].unique()

print(f"\nUnique patents with G06N CPC code: {len(ai_patent_ids)}")
print(f"Total unique patents queried: {cpc_data['patent_id'].nunique()}")

# Avoid division by zero
if cpc_data['patent_id'].nunique() > 0:
    print(f"AI patent percentage: {len(ai_patent_ids) / cpc_data['patent_id'].nunique() * 100:.2f}%")
else:
    print("AI patent percentage: N/A (no patents to analyze)")


Unique patents with G06N CPC code: 42
Total unique patents queried: 76494
AI patent percentage: 0.05%


In [119]:
# Show examples of G06N patents
if len(cpc_data) > 0 and 'is_ai_cpc' in cpc_data.columns and cpc_data['is_ai_cpc'].sum() > 0:
    print("\nSample G06N CPC codes:")
    print(cpc_data[cpc_data['is_ai_cpc']][['patent_id', 'cpc_group']].head(20))
else:
    print("\nNo G06N CPC codes found in the dataset.")


Sample G06N CPC codes:
       patent_id  cpc_group
311821   7433853   G06N5/04
311825   7433853   G06N7/01
423999   8145590   G06N5/04
424003   8145590   G06N7/01
470137   8583582  G06N5/048
499818   8740789   G06N3/02
499819   8740789  G06N3/042
630286   9959940  G06N3/084
630287   9959940  G06N3/086
630288   9959940   G06N5/02
630289   9959940  G06N20/00
634278   9737227  G06N5/047
651505   9974742  G06N20/00
651506   9974742  G06N3/126
651507   9974742   G06N5/01
651508   9974742  G06N5/025
651509   9974742  G06N5/045
651510   9974742   G06N7/01
651879   9792412  G06N3/084
651880   9792412  G06N3/086


## Step 6: Map AI Patents Back to GVKEYs

Identify which clinical trial firms (GVKEYs) hold at least one AI patent.

In [120]:
# Mark patents as AI in our patent dataset
ct_patents_with_gvkey['is_ai_patent'] = ct_patents_with_gvkey['patent_id'].astype(str).isin(ai_patent_ids)

print(f"Patents marked as AI: {ct_patents_with_gvkey['is_ai_patent'].sum()}")
print(f"Total patents: {len(ct_patents_with_gvkey)}")

Patents marked as AI: 42
Total patents: 77111


In [121]:
# Aggregate by GVKEY
# Flag firms that have at least one AI patent
if len(ct_patents_with_gvkey) > 0:
    firm_ai_status = ct_patents_with_gvkey.groupby('gvkey').agg({
        'patent_id': 'count',
        'is_ai_patent': 'sum'
    }).reset_index()

    firm_ai_status.columns = ['gvkey', 'total_patents', 'ai_patents']
    firm_ai_status['has_ai_patent'] = firm_ai_status['ai_patents'] > 0

    print(f"\nFirms with patents: {len(firm_ai_status)}")
    print(f"Firms with AI patents: {firm_ai_status['has_ai_patent'].sum()}")

    print("\nSample firm AI status:")
    print(firm_ai_status.head(20))
else:
    print("\nNo patents found for clinical trial firms. Creating empty firm_ai_status dataframe.")
    firm_ai_status = pd.DataFrame(columns=['gvkey', 'total_patents', 'ai_patents', 'has_ai_patent'])


Firms with patents: 311
Firms with AI patents: 4

Sample firm AI status:
     gvkey  total_patents  ai_patents  has_ai_patent
0   001259             20           0          False
1   001478           3320           0          False
2   001602           2025           0          False
3   002085            623           0          False
4   002222             85           0          False
5   002403           5260           0          False
6   003170           3219           6           True
7   003691             26           0          False
8   004632              3           0          False
9   004843             44           0          False
10  005020           1716           0          False
11  005241              8           0          False
12  006315             46           0          False
13  006730           4108           0          False
14  008530           6516           0          False
15  008762          13870          17           True
16  012180             95

## Step 7: Compute Final Statistics

Calculate the percentage of clinical trial firms with AI patents.

In [122]:
# Total clinical trial firms (from original data)
total_ct_firms = len(ct_gvkeys)

# Firms with at least one AI patent
if len(firm_ai_status) > 0:
    firms_with_ai = firm_ai_status['has_ai_patent'].sum()
else:
    firms_with_ai = 0

# Calculate percentage
percentage_with_ai = (firms_with_ai / total_ct_firms) * 100 if total_ct_firms > 0 else 0

print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)
print(f"\n1. Total number of clinical trial firms: {total_ct_firms}")
print(f"\n2. Number of firms with AI patents (G06N): {firms_with_ai}")
print(f"\n3. Percentage of firms with AI patents: {percentage_with_ai:.2f}%")
print("\n" + "=" * 70)

FINAL RESULTS

1. Total number of clinical trial firms: 673

2. Number of firms with AI patents (G06N): 4

3. Percentage of firms with AI patents: 0.59%



## Additional Analysis and Breakdown

In [123]:
# Breakdown of firms by patent activity
print("Breakdown of clinical trial firms:")
print(f"\n- Firms matched to DISCERN (with patents): {len(firm_ai_status)}")
print(f"- Firms not matched (no patents in DISCERN): {total_ct_firms - len(firm_ai_status)}")

if total_ct_firms > 0:
    print(f"\n- Match rate: {len(firm_ai_status) / total_ct_firms * 100:.2f}%")
else:
    print(f"\n- Match rate: N/A")

print(f"\nAmong firms with patents:")
print(f"- Firms with AI patents: {firms_with_ai}")
print(f"- Firms without AI patents: {len(firm_ai_status) - firms_with_ai}")

if len(firm_ai_status) > 0:
    print(f"- AI patent rate (among patenting firms): {firms_with_ai / len(firm_ai_status) * 100:.2f}%")
else:
    print(f"- AI patent rate (among patenting firms): N/A (no patenting firms found)")

Breakdown of clinical trial firms:

- Firms matched to DISCERN (with patents): 311
- Firms not matched (no patents in DISCERN): 362

- Match rate: 46.21%

Among firms with patents:
- Firms with AI patents: 4
- Firms without AI patents: 307
- AI patent rate (among patenting firms): 1.29%


In [124]:
# Distribution of AI patents per firm
if len(firm_ai_status) > 0 and firm_ai_status['has_ai_patent'].sum() > 0:
    print("\nDistribution of AI patents among firms with AI patents:")
    ai_firms = firm_ai_status[firm_ai_status['has_ai_patent']]
    print(ai_firms['ai_patents'].describe())

    print("\nTop 10 firms by number of AI patents:")
    top_ai_firms = ai_firms.nlargest(10, 'ai_patents')[['gvkey', 'total_patents', 'ai_patents']]
    top_ai_firms['ai_share'] = (top_ai_firms['ai_patents'] / top_ai_firms['total_patents'] * 100).round(2)
    print(top_ai_firms)
else:
    print("\nNo firms with AI patents found in the dataset.")


Distribution of AI patents among firms with AI patents:
count     4.000000
mean     10.500000
std       7.047458
min       3.000000
25%       5.250000
50%      11.000000
75%      16.250000
max      17.000000
Name: ai_patents, dtype: float64

Top 10 firms by number of AI patents:
      gvkey  total_patents  ai_patents  ai_share
15   008762          13870          17      0.12
112  025279          13139          16      0.12
6    003170           3219           6      0.19
22   013786             69           3      4.35


In [125]:
# Merge with clinical trial data to see firm names
if len(firm_ai_status) > 0:
    ct_firm_info = clinical_trials[['gvkey_sponsor', 'sponsor_name']].drop_duplicates()
    ct_firm_info.columns = ['gvkey', 'sponsor_name']

    firm_ai_status_with_names = firm_ai_status.merge(
        ct_firm_info,
        on='gvkey',
        how='left'
    )

    print("\nTop 10 firms by number of AI patents (with names):")
    top_firms_with_names = firm_ai_status_with_names.nlargest(10, 'ai_patents')
    print(top_firms_with_names[['gvkey', 'sponsor_name', 'total_patents', 'ai_patents', 'has_ai_patent']])
else:
    print("\nNo firms with patents found. Cannot display firm rankings.")
    # Create empty dataframe to avoid errors in later cells
    firm_ai_status_with_names = pd.DataFrame(columns=['gvkey', 'sponsor_name', 'total_patents', 'ai_patents', 'has_ai_patent'])


Top 10 firms by number of AI patents (with names):
      gvkey                                      sponsor_name  total_patents  \
15   008762                                Procter and Gamble          13870   
116  025279                     Boston Scientific Corporation          13139   
6    003170                                 Colgate Palmolive           3219   
23   013786                                Heron Therapeutics             69   
0    001259                         Tamir Biotechnology, Inc.             20   
1    001478  Wyeth is now a wholly owned subsidiary of Pfizer           3320   
2    001602                                             Amgen           2025   
3    002085                        Bausch & Lomb Incorporated            623   
4    002222                           Savient Pharmaceuticals             85   
5    002403                              Bristol-Myers Squibb           5260   

     ai_patents  has_ai_patent  
15           17           True  
1

## Export Results

In [126]:
# Export firm-level results
if len(firm_ai_status) > 0:
    output_path = Path('clinical_trial_firms_ai_patents.csv')
    firm_ai_status_with_names.to_csv(output_path, index=False)
    print(f"\nFirm-level results exported to: {output_path}")
    
    # Calculate AI patent rate safely
    total_patents_sum = firm_ai_status['total_patents'].sum()
    ai_patents_sum = firm_ai_status['ai_patents'].sum()
    ai_patent_rate = f"{ai_patents_sum / total_patents_sum * 100:.2f}%" if total_patents_sum > 0 else "N/A"
    
    # Export summary statistics
    summary_stats = pd.DataFrame({
        'Metric': [
            'Total clinical trial firms',
            'Firms with AI patents (G06N)',
            'Percentage with AI patents',
            'Firms matched to DISCERN',
            'Match rate',
            'Total patents from CT firms',
            'Total AI patents',
            'AI patent rate'
        ],
        'Value': [
            total_ct_firms,
            firms_with_ai,
            f"{percentage_with_ai:.2f}%",
            len(firm_ai_status),
            f"{len(firm_ai_status) / total_ct_firms * 100:.2f}%",
            total_patents_sum,
            ai_patents_sum,
            ai_patent_rate
        ]
    })

    summary_path = Path('summary_statistics.csv')
    summary_stats.to_csv(summary_path, index=False)
    print(f"Summary statistics exported to: {summary_path}")
else:
    print("\nWarning: No firms matched to DISCERN. Cannot export results.")
    print("This may indicate a data matching issue. Please check:")
    print("1. GVKEY format compatibility between clinical trials and DISCERN")
    print("2. Fiscal year range (2000-2021) may be too restrictive")
    print("3. Clinical trial firms may not be in DISCERN patent database")

print("\n" + "="*70)
print("ANALYSIS COMPLETE")
print("="*70)


Firm-level results exported to: clinical_trial_firms_ai_patents.csv
Summary statistics exported to: summary_statistics.csv

ANALYSIS COMPLETE


## Summary

This notebook:
1. Loaded clinical trial data and extracted unique GVKEYs
2. Mapped GVKEYs to permno_adj using DISCERN firm panel (2000-2021)
3. Retrieved patent IDs for these firms from DISCERN patent grant data
4. Queried PatentsView API for CPC classifications
5. Filtered for AI patents (G06N CPC prefix)
6. Computed the percentage of clinical trial firms with AI patents

**Key Findings:**
- Total clinical trial firms: [Computed above]
- Firms with AI patents: [Computed above]
- Percentage: [Computed above]

---

**Data Sources:**
- DISCERN 2.0.1: https://zenodo.org/records/13619821
- PatentsView API: https://api.patentsview.org/
- Clinical trial sample: Provided dataset

**CPC G06N Definition:**
- G06N: Computing arrangements based on specific computational models
  - G06N3: Neural networks
  - G06N5: Knowledge-based models  
  - G06N7: Probabilistic/fuzzy logic
  - G06N10: Quantum computing
  - G06N20: Machine learning